In [8]:
import os
from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaLLM
from langchain_ollama import OllamaEmbeddings
from langchain_ibm import ChatWatsonx
from langchain_ibm import WatsonxEmbeddings
from langchain.agents import create_agent
import langchain_core.tools
import langchain_core.runnables
import langchain_core.prompts
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\soldesk\AppData\Local\Temp\ipykernel_12436\1737910632.py:14: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [6]:
from dotenv import load_dotenv


load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    max_tokens=2000,
    temperature=0,
)

watson_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url=f"{watsonx_ai_url}",
    api_key=f"{apikey}",
    project_id=f"{project_id}",
    params={"temperature": 0},
)


# RouterChain (LCEL Router : 조건부 분기 패턴)
#### - Router 패턴 : 입력에 따라 적절한 체인 or Runnable로 분기
#### - 분류기가 입력을 분석 -> 적절한 체인으로 라우팅
#### - ex) 질문 유형 (코딩, 수학, 일반 ...)에 따라 다른 전문 프롬프트 적용
#### - 동작 흐름: 1. 입력 -> 2. 분류 -> 3. 라우팅 -> 4. 실행 -> 5. 반환

In [ ]:


parser = StrOutputParser()

math_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "당신은 수학 전문가입니다. 풀이 과정을 단계별로 설명하세요."),
            ("human", "{question}"),
        ]
    )
    | watson_llm
    | parser
)

code_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "당신은 시니어 개발자입니다. 코드와 주석을 함께 제공하세요."),
            ("human", "{question}"),
        ]
    )
    | watson_llm
    | parser
)

chat_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "당신은 친절한 AI 어시스턴트입니다. 한국어로 답변하세요."),
            ("human", "{question}"),
        ]
    )
    | watson_llm
    | parser
)

# 질문을 읽고 유형 반환
classify_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "사용자 질문을 분류하세요. 반드시 math, code, general 중 하나만 출력하세요.",
            ),
            ("human", "{question}"),
        ]
    )
    | watson_llm
    | parser
)

# 라우터 함수
def route(inputs: dict):
    category = classify_chain.invoke(inputs).strip().lower()
    print(f"=> 분류 결과: {category}")

    if "math" in category:
        return math_chain
    if "code" in category:
        return code_chain
    return chat_chain


router_chain = RunnableLambda(lambda x: route(x).invoke(x))

In [15]:
questions = [
    {
        'question' : '피타고라스 정리를 증명해라'
    },
    {
        'question' : '오늘 저녁 메뉴 추천해줘'
    },
    {
        'question' : 'python으로 버블 정렬 구현해줘'
    }
]

for q in questions:
    print(f"Q : {q['question']}")
    print(f"A : {router_chain.invoke(q)[:100]}...\n")

Q : 피타고라스 정리를 증명해라
=> 분류 결과: math
A : 물론, 피타고라스의 정리를 증명하는 것에 대해 도와드리는 것이 기쁩니다. 피타고라스의 정리는 직각삼각형의 빗변의 길이의 제곱이 나머지 두 변의 길이의 제곱의 합과 같다는 것을 말합...

Q : 오늘 저녁 메뉴 추천해줘
=> 분류 결과: general
A : 저녁 메뉴로는 다양한 선택지가 있습니다. 건강하고 맛있는 한국 요리를 추천해 드리겠습니다.

1. 비빔밥: 밥에 다양한 채소와 고기를 넣고 고추장 소스를 섞어 먹는 한국의 대표적인...

Q : python으로 버블 정렬 구현해줘
=> 분류 결과: code
A : 물론입니다. 다음은 Python으로 구현한 버블 정렬 코드입니다. 주석을 포함하여 설명하겠습니다.
```python
def bubble_sort(arr):
    n = len(a...



In [18]:
from langchain_core.runnables import RunnableBranch

branch = RunnableBranch(
    (lambda x: "math" in x.get("topic", ""), math_chain),
    (lambda x: "code" in x.get("topic", ""), code_chain),
    (
        lambda x: "cooking" in x.get("topic", ""),
        ChatPromptTemplate.from_messages(
            [("system", "당신은 요리 전문가 입니다."), ("human", "{question}")]
        )
        | watson_llm
        | parser,
    ),
    chat_chain,
)

print(branch.invoke({"topic": "math", "question": "미분이란?"}))
print(branch.invoke({"topic": "cooking", "question": "김치찌개 레시피"}))
print(branch.invoke({"topic": "other", "question": "안녕하세요?"}))
print(branch.invoke({"topic": "code", "question": "파이썬으로 선택정렬 구현"}))

미분은 수학에서 함수의 변화율을 나타내는 개념입니다. 미분은 함수의 그래프에서 접선의 기울기를 구하는 과정으로, 함수의 변화율을 정확하게 파악할 수 있게 해줍니다. 미분은 미적분학의 핵심 개념 중 하나이며, 미분을 통해 함수의 최대값, 최소값, 변곡점 등을 찾을 수 있습니다.

미분의 과정을 단계별로 설명하겠습니다.

1. 함수 f(x)를 주어진다고 가정합니다.
2. 미분을 구하기 위해 독립변수 x에 대한 미소변화량을 나타내는 기호 'dx'를 사용합니다. 이때, 함수 값의 미소변화량을 'dy'로 나타냅니다.
3. 함수 f(x)의 미분은 다음과 같이 정의됩니다: dy/dx = f'(x) = lim(h→0) (f(x+h) - f(x))/h
4. 이 식에서 lim(h→0)은 h가 0에 가까워질 때의 극한을 의미합니다. 이 극한을 구하면 함수 f(x)의 미분, 즉 도함수 f'(x)를 얻을 수 있습니다.
5. 도함수 f'(x)는 원래 함수 f(x)의 변화율을 나타내며, 그래프에서 접선의 기울기를 구하는 데 사용됩니다.

예를 들어, 함수 f(x) = x^2를 미분해보겠습니다.

1. f(x) = x^2
2. dy = f(x+dx) - f(x) = (x+dx)^2 - x^2
3. dy/dx = lim(h→0) ((x+h)^2 - x^2)/h
4. 이 식을 정리하면, dy/dx = lim(h→0) (x^2 + 2xh + h^2 - x^2)/h = lim(h→0) (2xh + h^2)/h
5. h를 제외한 항으로 나누면, dy/dx = lim(h→0) (2x + h) = 2x

따라서, 함수 f(x) = x^2의 미분은 f'(x) = 2x입니다. 이 도함수는 x에 대한 함수 f(x)의 변화율을 나타내며, 그래프에서 접선의 기울기를 구하는 데 사용됩니다.
김치찌개는 한국 요리의 대표적인 한 그릇으로, 매콤하고 깊은 맛이 특징인 요리입니다. 다음은 기본적인 김치찌개 레시피입니다.

### 재료:
- 김치: 2컵 (배추김치가 가장 일반적이지만, 다른 종류의 김치도 사용 가능)
-

### SequentialChain (단계별 순차 파이프라인)

- 앞 단계 출력이 뒷단계의 입력으로 흘러 들어감
- | => SequentialChain
- .assign() : 중간 결과를 보존하며 여러 단계를 누적할 수 있음
- 각 단계의 출력 타입과 입력 타입이 일치 해야함 

In [19]:
# 체인 정의

from langchain_core.runnables import RunnablePassthrough


translate_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

summarize_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 3문장으로 번역하세요: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

sentiment_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 답하세요.: \n{text}",
            ),
        ]
    )
    | watson_llm
    | parser
)

report_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "아래 분석 결과를 바탕으로 한 줄 최종 보고서를 작성하세요: \n"
                "원문 : {text} \n 번역 : {translated}\n 요약:{summary}\n 감정: {sentiment}",
            ),
        ]
    )
    | watson_llm
    | parser
)

# assign() : 단계별 결과 누적
pipeline = (
    RunnablePassthrough.assign(translated=translate_chain)
    .assign(summary=summarize_chain)
    .assign(sentiment=sentiment_chain)
    .assign(report=report_chain)
)

result = pipeline.invoke({
    "text" : "python is a versatile language loved by developers worldwide."
})

print("번역 : ", result['translated'])
print("요약 : ", result['summary'])
print("감정 : ", result['sentiment'])
print("보고서 : ", result['report'])

번역 :  파이썬은 개발자들에게 사랑받는 다재다능한 언어입니다.
요약 :  파이썬은 개발자들에게 사랑받는 다재다능한 언어입니다.
감정 :  긍정
보고서 :  파이썬은 개발자들에게 사랑받는 다재다능한 언어입니다.


In [21]:
# 조건부 단계 삽입

detect_lang_chain = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트의 언어를 korean/english/other 중 하나로만 답하세요: \n {text}"),
]) | watson_llm | parser

def maybe_translate(inputs):
    lang = detect_lang_chain.invoke(inputs).strip().lower()

    if 'english' in lang or 'other' in lang:
        translated = translate_chain.invoke(inputs)
        return {**inputs, 'text': translated, 'was_translated':True}
    return {**inputs, 'was_transated':False}

smart_pipeline = (RunnableLambda(maybe_translate)
                  | RunnablePassthrough.assign(summary = summarize_chain)
                  | RunnablePassthrough.assign(sentiment = sentiment_chain))

r1 = smart_pipeline.invoke({'text':'Python is great'})
r2 = smart_pipeline.invoke({'text':'파이썬은 최고야!'})

print(r1['was_translated'], r1['summary'][:50])
print(r2['was_translated'], r2['summary'][:50])

True I really love Python.


KeyError: 'was_translated'

### MapReduceChain(대용량 문서 처리)
- Map : 문서를 청크로 나눠 각각 처리(요약, 추출..) => 병렬 실행 가능
- Reduce : Map 결과들을 하나로 합쳐 최종 답변 생성

In [ ]:


map_chain = ChatPromptTemplate.from_messages([
    ("system", "다음 텍스트를 2문장으로 요약하세요."),
    ("user", "{chunk}"),
]) | watson_llm | parser

reduce_chain = ChatPromptTemplate.from_messages([
    ("system", "다음은 긴 문서의 섹션별 요약입니다. 전체를 5문장으로 통합 요약하세요."),
    ("user", "{summaries}"),
]) | watson_llm | parser


def map_reduce_summarize(file_path):
    loader = PyMuPDFLoader(file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(loader.load())
    print(f"chunks : {len(chunks)}")

    chunk_inputs = [{'chunk' : c.page_content} for c in chunks]
    summaries = map_chain.batch(chunk_inputs, config={'max_concurrency': 5})
    print(f"총 청크 수 {len(summaries)} 개 요약 생성")

    #---- Reduce : 요약 통합
    # 문서 결합
    combined = "\n\n".join(f"[섹션 {i+1}] {s}" for i,s in enumerate(summaries))
    final = reduce_chain.invoke({'summaries': combined})
    return final

result = map_reduce_summarize("./data/2026 상 삼성전자 DX부문 직무기술서.pdf")
print(result)



chunks : 35
총 청크 수 35 개 요약 생성
삼성전자의 다양한 사업부에서는 회로 개발 및 설계, 통신 시스템 최적화, AI 기반 소프트웨어 및 하드웨어 개발, 네트워크 자동화 솔루션, 신제품 신뢰성 검증, 마케팅 전략 수립, 구매 프로세스 혁신, 환경안전 관리 등 다양한 기술과 전문 분야에서 인재를 모집하고 있습니다. 각 사업부는 전자공학, 컴퓨터공학, 소프트웨어 개발, 마케팅, 구매, 환경안전 등 전공에 맞는 인재를 필요로 하며, 글로벌 시장에서의 경쟁력 강화와 혁신적인 가치 창출을 목표로 합니다.


In [9]:
# map_reduce + QA

parser = StrOutputParser()

# 질문과 관련된 정보 추출
extract_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트에서 질문과 관련된 정보만 추출하세요. 없으면 '없음' 으로 답하세요"),
            ("user", "질문 : {question}\n\n 텍스트:\n{chunk}"),
        ]
    )
    | watson_llm
    | parser
)

# 답변
answer_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 추출된 정보를 바탕으로 질문에 답하세요."),
            ("user", "질문 : {question} \n\n 추출 정보:\n {extracts}"),
        ]
    )
    | watson_llm
    | parser
)

chunks = None

def extract_chunks(file_path):
    global chunks

    loader = PyMuPDFLoader(file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(loader.load())
    print(f"총 청크 수 {len(chunks)}")


def map_reduce_qa(chunks, question):
    if not chunks:
        return "문서 청크가 없습니다. 먼저 extract_chunks()를 실행하세요."

    # Map
    inputs = [{"question" : question, "chunk": c.page_content} for c in chunks]
    extracts = extract_chain.batch(inputs, config={"max_concurrency": 5})

    # '없음' 으로 넘어온 청크 필터링
    relevant = [e for e in extracts if e.strip() and e.strip() != '없음']
    print(f"관련 청크 : {len(relevant)} / {len(chunks)}")

    if not relevant:
        return "문서에서 질문과 관련된 정보를 찾지 못했습니다."

    # Reduce
    combined = "\n\n".join(relevant)
    return answer_chain.invoke({'question': question, 'extracts':combined})

extract_chunks('./data/2026 상 삼성전자 DX부문 직무기술서.pdf')
result = map_reduce_qa(chunks, '삼성전자 DX부문에서 AI 관련 직무는 어떤 역할을 하나요?')
print(result)

총 청크 수 35
관련 청크 : 12 / 35
삼성전자 DX부문에서 AI 관련 직무는 다음과 같은 역할을 수행합니다:

1. AI 기반 네트워크 자동화 솔루션 개발: AI 기술을 활용하여 네트워크 관리 및 운영을 자동화하는 솔루션을 개발합니다.
2. AI 기반 rAPP 개발: 이상 탐지, 에너지 절감, 자원 할당 등의 기능을 수행하는 AI 기반 rAPP(무선 애플리케이션)을 개발합니다.
3. Cloud Native 기반 가상화 네트워크 운영 솔루션 개발: 클라우드 네이티브 기술을 활용하여 가상화된 네트워크를 운영하는 솔루션을 개발합니다.
4. SDN 개발: 소프트웨어 정의 네트워킹(SDN) 기술을 활용하여 네트워크 관리 및 운영을 효율적으로 수행하는 솔루션을 개발합니다.
5. AI 검증 및 온보딩동작자동화: AI 검증 및 온보딩 동작을 자동화하는 플랫폼을 구축하고, Python을 활용하여 자동화 테스트를 수행합니다.
6. 품질관련서비스 실행: AI 관련 품질 관련 서비스를 실행하고, 플랫폼을 통해 효율적인 관리를 수행합니다.

이러한 역할들은 AI 기술을 활용하여 네트워크 관리 및 운영을 효율적으로 수행하고, 고객에게 더 나은 서비스를 제공하는 것을 목표로 합니다.


#### ParallelChain (비동기 병렬 처리)
- RunnableParallel : 여러 체인을 동시에 실행하고 결과를 dict 결합
- 서로 독립적인 작업을 동시에 처리해 시간을 절약
- batch()
- 독립적인 LLM 호출이 3개 였고 순차 호출이 30초가 걸렸다면 병렬 10초로 단축

In [13]:
from joblib import parallel
import time

from langchain_core.runnables import RunnableParallel

translate_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 한국어로 번역하세요. 번역문만 출력: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

summarize_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "다음 텍스트를 3문장으로 번역하세요: \n{text}"),
        ]
    )
    | watson_llm
    | parser
)

keyword_chain = (
    ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "키워드 5개 추출 : \n{text}",
            ),
        ]
    )
    | watson_llm
    | parser
)

text = {
    "text": """Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation,[38] "plain English" naming, an extensive ("batteries-included") standard library, and garbage collection. Python supports multiple programming paradigms but with an emphasis on object-oriented programming and dynamic typing.

Guido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5,[39] capabilities and keywords for typing were added to the language, allowing optional static typing.[40] As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project's annual release cycle and five-year support policy. Python 3.15 is currently in the alpha development phase, and the stable release is expected to launch in October 2026.[41] Earlier versions in the 3.x series have reached end-of-life and no longer receive security updates.

Python has gained extensive use in the machine learning community.[42][43][44][45] It is widely taught as an introductory programming language.[46] Since 2003, Python has consistently ranked among the top ten most popular programming languages in the TIOBE Programming Community Index, which ranks programming languages based on searches across 24 platforms.[47]"""
}

start = time.time()
t = translate_chain.invoke(text)
s = summarize_chain.invoke(text)
k = keyword_chain.invoke(text)
print(f"순차 실행 : {time.time() - start:.1f}초")

# 병렬 실행
parallel = RunnableParallel(
    translated=translate_chain, summary=summarize_chain, keyword=keyword_chain
)

start = time.time()
result = parallel.invoke(text)
print(f"병렬 실행 :  {time.time() - start:.1f}초")

print("번역", result["translated"])
print("요약", result["summary"])
print("키워드", result["keyword"])

순차 실행 : 27.5초
병렬 실행 :  6.2초
번역 파이썬은 코드 가독성, 단순성, 그리고 중요한 들여쓰기를 사용하여 쉬운 작성을 강조하는 고급 일반 목적 프로그래밍 언어입니다.[38] "일반 영어" 명명법, 광범위한("배터리 포함") 표준 라이브러리, 그리고 가비지 수집을 사용합니다. 파이썬은 다양한 프로그래밍 패러다임을 지원하지만 객체 지향 프로그래밍과 동적 타이핑에 중점을 둡니다.

가이도 반 로섬은 1980년대 말에 ABC 프로그래밍 언어의 후속으로 파이썬 작업을 시작했습니다. 2008년에 출시된 파이썬 3.0은 주요 개정판이었으며 이전 버전과 완전히 하위 호환되지 않았습니다. 파이썬 3.5부터[39] 언어에 타이핑 기능과 키워드가 추가되었으며, 이를 통해 선택적 정적 타이핑이 가능해졌습니다.[40] 2026년 기준, 파이썬 소프트웨어 재단은 파이썬 3.10, 3.11, 3.12, 3.13, 3.14를 지원하고 있으며, 이는 프로젝트의 연간 출시 주기와 5년 지원 정책을 따릅니다. 파이썬 3.15는 현재 알파 개발 단계에 있으며, 안정적인 출시는 2026년 10월에 예상됩니다.[41] 3.x 시리즈의 이전 버전은 수명이 다했으며 더 이상 보안 업데이트를 받지 않습니다.

파이썬은 머신 러닝 커뮤니티에서 광범위하게 사용되고 있습니다.[42][43][44][45] 파이썬은 입문용 프로그래밍 언어로 널리 가르치고 있습니다.[46] 2003년 이후 파이썬은 TIOBE 프로그래밍 커뮤니티 지수에서 상위 10개 프로그래밍 언어 중 하나로 지속적으로 순위를 유지해 왔으며, 이 지수는 24개 플랫폼에서 검색을 기반으로 프로그래밍 언어를 순위화합니다.[47]
요약 파이썬은 코드 가독성, 단순성, 그리고 중괄호를 사용한 중요한 들여쓰기를 통해 쉬운 작성을 강조하는 고급 일반 목적 프로그래밍 언어입니다. 파이썬은 객체 지향 프로그래밍과 동적 타이핑에 중점을 두고 다양한 프로그래밍 패러다임을 지원하며, "배터리 포함" 표준 라이브러리와 가비지 수집을 갖추고 있습니다. 파

In [14]:
# 병렬로 만들어진 체인을 비동기식 처리

import asyncio


async def analyze_multiple(texts):
    tasks = [parallel.ainvoke({"text": t}) for t in texts]
    results = await asyncio.gather(*tasks)
    return results


texts = [
    "Python is great for data science.",
    "Javascript dominates frontend development",
    "Rust is known for memory safety",
]

start = time.time()
results = await analyze_multiple(texts)
print(f"비동기 병렬 {len(texts)} 개: {time.time() - start:.1f}초")

for i, r in enumerate(results):
    print(f"[{i + 1}] 번역 : {r['translated'][:40]}...")

results = summarize_chain.batch(
    [{"text": t} for t in texts * 3], config={"max_concurrency": 3}
)

비동기 병렬 3 개: 1.8초
[1] 번역 : 파이썬은 데이터 사이언스에 훌륭합니다....
[2] 번역 : 자바스크립트가 프론트엔드 개발을 지배하고 있습니다....
[3] 번역 : 러스트는 메모리 안전성으로 유명합니다....
